# _lib/staging_cdf — staging Change Data Feed reader

Wrapper met één functie rond `spark.readStream.option("readChangeFeed", "true")`
met consistente opties.  Herbruikbaar voor alle Silver-tabellen die staging CDF
consumeren.

**Importeer via `%run ./_lib/staging_cdf` vanuit het Silver DLT-pipeline-notebook.**

### Waarom CDF + apply_changes?

staging schrijft in twee modes: `full` (overwrite) en `incremental` (append).  Een
gewone streaming-read op een tabel die overschreven wordt faalt met een
schema-evolution- of source-not-streamable-fout.  CDF zet overwrites om in
`delete_row` + `insert_row`-events, en DLT's `apply_changes` MERGEt die declaratief
naar Silver — geen Silver-pipeline-state breekt bij een staging-mode-switch.  Zie ADR 0002.

In [ ]:
from pyspark.sql import DataFrame


def read_staging_cdf(table: str) -> DataFrame:
    """Geeft een streaming DataFrame terug die leest uit het Change Data Feed van een staging-tabel.

    Args:
        table: Volledig gekwalificeerde tabelnaam, bijv. 'DEMO.STAGING_AZURESTORAGE.order_header'

    Returns:
        Een streaming DataFrame met CDF-kolommen (_change_type, _commit_version,
        _commit_timestamp) plus alle bronkolommen.

    De aanroeper (DLT apply_changes) selecteert de relevante kolommen uit deze stream
    en routeert ze op basis van _change_type naar de Streaming Table-doeltabel.
    """
    return (
        spark.readStream
        .option("readChangeFeed", "true")
        .option("startingVersion", 0)
        .table(table)
    )


print("staging_cdf geladen: read_staging_cdf")